# EchoStage: Adaptive Virtual Stage Director

A companion notebook for exploring EchoStage's **bounded stage-cue decision pattern**. EchoStage represents the next stage action as one of five typed choices: `HOLD`, `INTENSIFY`, `CONTRAST`, `SOFTEN`, or `TRANSITION`.

**Important honesty note:** this notebook can call a deployed EchoStage API, but it does not itself load Strands Decider 2B. The API response's `source` field tells you whether a decision came from `strands-decider-2b`, `mock-policy`, or a fallback. Use real inference for the hackathon demonstration; mock mode is for development only. Scene audience engagement is simulated, not measured from real people.

## Decision flow
1. Observe the current scene state.
2. Ask the decision model to select one typed stage cue.
3. Apply a confidence threshold and deterministic safety constraints.
4. Fall back to `HOLD` when confidence is low, the decision is invalid, or the model is unavailable.
5. Use Amazon Bedrock separately to generate a creative scene brief.


In [ ]:
# Configure the URL of your deployed EchoStage FastAPI backend.
# Example: https://echostage-api.onrender.com
API_BASE = "https://YOUR-BACKEND-URL.onrender.com"
API_BASE = API_BASE.rstrip("/")

# If requests is not installed in this notebook environment, run:
# %pip install requests
import requests
from pprint import pprint

print("API base:", API_BASE)


## 1. Check service and decision-model status

The health endpoint reports whether the API is responding and whether the configured Decider endpoint is available. A health result with `mode: mock` is not live Decider inference.

In [ ]:
health_response = requests.get(f"{API_BASE}/health", timeout=20)
health_response.raise_for_status()
health = health_response.json()
pprint(health)

decider_status = health.get("decider", {})
print("\nDecider available:", decider_status.get("available"))
print("Decider mode:", decider_status.get("mode", "check details/source"))


## 2. Inspect available scenarios

EchoStage ships with a suspense scenario and a reflective-monologue scenario. The engagement value is a simulated input signal for the prototype.

In [ ]:
scenarios_response = requests.get(f"{API_BASE}/api/scenarios", timeout=20)
scenarios_response.raise_for_status()
scenarios = scenarios_response.json()
for scenario in scenarios:
    print(f"{scenario['id']}: {scenario['name']} — {scenario['description']}")
    pprint(scenario['initial_state'])
    print("-" * 72)


## 3. Submit a typed stage-cue decision

The request supplies a structured scene state. The API returns the requested action, executed action, confidence (when available), latency, source, fallback information, and updated state. The server-side policy remains responsible for enforcing constraints.

In [ ]:
scene_state = {
    "scene_id": "suspense",
    "scene_phase": "rising_tension",
    "narrative_progress": 0.42,
    "stage_intensity": 0.58,
    "simulated_engagement": 0.64,
    "pacing_pressure": 0.48,
    "current_lighting": "deep_blue",
    "last_action": "HOLD",
    "seconds_since_last_transition": 30,
    "transition_allowed": True
}

decision_response = requests.post(
    f"{API_BASE}/api/decide",
    json={"state": scene_state},
    timeout=45,
)
decision_response.raise_for_status()
decision = decision_response.json()
pprint(decision)


In [ ]:
print("Requested action:", decision.get("requested_action"))
print("Executed action:", decision.get("executed_action"))
print("Confidence:", decision.get("confidence"))
print("Threshold:", decision.get("confidence_threshold"))
print("Source:", decision.get("source"))
print("Fallback reason:", decision.get("fallback_reason"))
print("Latency (ms):", decision.get("latency_ms"))

if decision.get("source") != "strands-decider-2b":
    print("\nNOTE: This response is not confirmed as live Strands Decider 2B inference.")


## 4. Generate a scene brief with Amazon Bedrock

Decider chooses a bounded action; Bedrock handles the generative writing task. This separation keeps the control decision typed and inspectable while allowing the creative brief to be free-form text. This call succeeds only if the backend has valid AWS credentials, model access, region configuration, and IAM permissions.

In [ ]:
brief_response = requests.post(
    f"{API_BASE}/api/scene-brief",
    json={
        "theme": "A detective realizes the clock has stopped just before the reveal",
        "style": "cinematic theatre"
    },
    timeout=60,
)

if brief_response.ok:
    brief = brief_response.json()
    print("Model:", brief.get("model_id"))
    print("\nScene brief:\n")
    print(brief.get("brief"))
else:
    print("Scene-brief request failed:", brief_response.status_code)
    print(brief_response.text)
    print("Check Bedrock model access, AWS_REGION, credentials, and IAM permissions on the backend.")


## 5. Inspect recent decision events

This endpoint is useful for observing decision sources and fallbacks during a demo. The current prototype writes to local JSONL storage, so history may not persist across container restarts unless persistent storage is configured.

In [ ]:
events_response = requests.get(f"{API_BASE}/api/events?limit=10", timeout=20)
events_response.raise_for_status()
events_payload = events_response.json()
pprint(events_payload)


## Reusable decision pattern

**Pattern tag:** `#policy`

- **Typed question:** Which single stage action should happen next?
- **Options:** `HOLD`, `INTENSIFY`, `CONTRAST`, `SOFTEN`, `TRANSITION`.
- **Inputs:** scene phase, narrative progress, stage intensity, simulated engagement, pacing pressure, current lighting, prior action, and transition constraints.
- **Wiring:** the backend's `DeciderAdapter` requests the typed choice; `decision_policy.py` applies the confidence threshold and deterministic constraints before executing the action.
- **Low-confidence handling:** execute `HOLD` when confidence is below the configured threshold.
- **Unavailable model handling:** fall back to `HOLD` and label the source as fallback.
- **Reuse:** replace the stage state and action options with your own domain state and allowed actions, preserve the confidence/fallback layer, and document the typed question and option meanings.

## Before using this as hackathon evidence

1. Replace `API_BASE` with your real deployed backend URL.
2. Run the notebook against the deployed service.
3. Verify the decision response's `source` is `strands-decider-2b` for a real inference demonstration.
4. Verify the scene brief was generated by Amazon Bedrock.
5. Publish this notebook in the public repository. Do not publish credentials, access tokens, or private endpoints.

Official submission requires a public repository, a published AWS Builder Center article, a public demo video under three minutes, and the tags `#strands-decider-2b` plus one pattern tag. This notebook is supporting documentation, not a substitute for those items.